# 📋 Day 04: Topological Sort — Ordering Dependencies

---

## What Is Topological Sort?

A linear ordering of vertices in a DAG (Directed Acyclic Graph) such that for every edge u → v, u comes before v.

Think of it as: **which order should I take courses** if some have prerequisites?

```
Prerequisites: Math → Physics → Engineering
                      Math → CS
Topo order:    Math, Physics, CS, Engineering (or Math, CS, Physics, Engineering)
```

**Two approaches:** Kahn's algorithm (BFS with in-degrees) and DFS-based.

---

In [ ]:
from collections import defaultdict, deque

## 🎯 Problem 1: Course Schedule II (LC #210)

Return the order to take courses. If impossible (cycle), return empty.

### Kahn's Algorithm (BFS)

**Idea:** Start with nodes that have no prerequisites (in-degree = 0). Process them, reduce neighbors' in-degrees. When a neighbor's in-degree hits 0, add it to the queue.

If we process all nodes, we have a valid ordering. If not, there's a cycle.

In [ ]:
def findOrder(numCourses, prerequisites):
    """
    Kahn's algorithm: BFS with in-degrees.
    Time: O(V + E), Space: O(V + E)
    """
    graph = defaultdict(list)
    in_degree = [0] * numCourses
    
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        in_degree[course] += 1
    
    # Start with courses that have no prerequisites
    queue = deque()
    for i in range(numCourses):
        if in_degree[i] == 0:
            queue.append(i)
    
    order = []
    while queue:
        course = queue.popleft()
        order.append(course)
        for next_course in graph[course]:
            in_degree[next_course] -= 1
            if in_degree[next_course] == 0:
                queue.append(next_course)
    
    # If all courses processed → valid order, else cycle exists
    return order if len(order) == numCourses else []

print(findOrder(4, [[1,0],[2,0],[3,1],[3,2]]))  # [0, 1, 2, 3] or [0, 2, 1, 3]
print(findOrder(2, [[1,0],[0,1]]))                # [] (cycle!)

**Why Kahn's is preferred in interviews:** It's intuitive (remove nodes with no dependencies), naturally detects cycles (leftover nodes = cycle), and gives the ordering directly.

---

## 🎯 Problem 2: Alien Dictionary (LC #269)

Given sorted words in an alien language, derive the character ordering.

```
Words: ["wrt", "wrf", "er", "ett", "rftt"]
From comparing adjacent words:
  wrt vs wrf → t comes before f
  wrf vs er  → w comes before e
  er vs ett  → r comes before t
  ett vs rftt → e comes before r

Order: w → e → r → t → f
```

**Approach:** Extract edges by comparing adjacent words, then topological sort.

In [ ]:
def alienOrder(words):
    """
    Extract ordering from adjacent word pairs, then topo sort.
    Time: O(total chars), Space: O(unique chars)
    """
    # Collect all unique characters
    graph = {ch: set() for word in words for ch in word}
    in_degree = {ch: 0 for ch in graph}
    
    # Compare adjacent words to extract edges
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i + 1]
        # Edge case: ["abc", "ab"] is invalid
        if len(w1) > len(w2) and w1[:len(w2)] == w2:
            return ""  # Invalid ordering
        for c1, c2 in zip(w1, w2):
            if c1 != c2:
                if c2 not in graph[c1]:
                    graph[c1].add(c2)
                    in_degree[c2] += 1
                break  # Only first difference matters
    
    # Kahn's topological sort
    queue = deque([ch for ch in in_degree if in_degree[ch] == 0])
    result = []
    
    while queue:
        ch = queue.popleft()
        result.append(ch)
        for neighbor in graph[ch]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    if len(result) < len(graph):
        return ""  # Cycle = invalid
    return "".join(result)

print(alienOrder(["wrt", "wrf", "er", "ett", "rftt"]))  # "wertf"
print(alienOrder(["z", "x"]))                             # "zx"
print(alienOrder(["z", "x", "z"]))                        # "" (cycle)

---

## 🗺️ Topological Sort Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TOPOLOGICAL SORT PATTERNS                                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  KAHN'S (BFS):                                                      ║
║    1. Compute in-degrees                                            ║
║    2. Queue all nodes with in-degree 0                              ║
║    3. Process queue: decrement neighbors, enqueue when 0            ║
║    4. If all processed → valid order. Else → cycle.                ║
║                                                                      ║
║  TRIGGER WORDS: "order", "prerequisites", "dependencies",          ║
║  "schedule", "sequence", "before/after"                            ║
║                                                                      ║
║  ONLY WORKS ON DAGs (Directed Acyclic Graphs)                       ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 210 | Course Schedule II | Medium | Kahn's algorithm |
| 269 | Alien Dictionary | Hard | Extract edges + topo sort |
| 310 | Minimum Height Trees | Medium | Peel leaves inward (reverse topo) |
| 1136 | Parallel Courses | Medium | Topo sort + level tracking |
| 2115 | Find All Recipes | Medium | Topo sort with string nodes |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Topo sort = ordering where dependencies come first             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Kahn's: BFS from in-degree 0 nodes. Natural cycle detection.  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 See "prerequisites/dependencies/order" → think topo sort       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Only works on DAGs. Cycle = no valid ordering exists.          │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Union-Find

Disjoint sets — the O(1) way to track connected components.